## Transport problem and lattice Boltzmann discretization

We consider the rectangular domain

$$
\Omega=(a_1,b_1)\times\cdots\times(a_d,b_d)\subset\mathbb{R}^d,
\qquad d\in\{1,2,3\},
$$

and the time interval $I_T=(0,T)$. We study the linear transport equation

$$
\frac{\partial f}{\partial t}+\mathbf{u}\cdot\nabla f=0,
$$

where $f(\mathbf{x},\mathbf{u},t)$ denotes the particle distribution function and $\mathbf{u}$ the transport velocity. The initial condition is prescribed by

$$
f(\mathbf{x},\mathbf{u},0)=f_0(\mathbf{x},\mathbf{u}),
\qquad \mathbf{x}\in\Omega.
$$

This corresponds to the collision-free transport part of the Boltzmann equation. Collision and external-force terms are omitted here, resulting in the reversible transport model considered in this tutorial.

### Lattice Boltzmann discretization

The lattice Boltzmann method (LBM) discretizes the velocity space using a finite set

$$
\mathbf{c}_i=(c_{i1},\ldots,c_{id})^T\in\mathbb{R}^d,
\qquad i\in\{1,\ldots,q\}.
$$

The corresponding velocity sets are commonly denoted by **DdQq**, where $d$ is the spatial dimension and $q$ the number of discrete velocities. Common examples are D1Q3, D2Q9, and D3Q27.
<p align="center">
  <img src="./discrete_velocity_sets.png" width="700">
</p>

For each discrete velocity $\mathbf{c}_i$, we introduce the distribution

$$
f_i(\mathbf{x},t)=f(\mathbf{x},\mathbf{c}_i,t),
$$

which satisfies

$$
\frac{\partial f_i}{\partial t}
+\mathbf{c}_i\cdot\nabla f_i=0.
$$

Macroscopic quantities can be recovered from moments of these distributions. In particular,

$$
\rho(\mathbf{x},t)\approx\sum_{i=1}^{q}f_i(\mathbf{x},t),
$$

and

$$
\rho(\mathbf{x},t)\mathbf{u}(\mathbf{x},t)
\approx
\sum_{i=1}^{q}\mathbf{c}_i f_i(\mathbf{x},t).
$$

To solve the transport equations numerically, the spatial domain is discretized by a regular lattice $\Omega_{\Delta x}\subset\overline{\Omega}$ with spacing $\Delta x$ and time step $\Delta t$. Applying the method of characteristics gives the **streaming relation**

$$
f_i(\mathbf{x}+\mathbf{c}_i\Delta t,t+\Delta t)
=
f_i(\mathbf{x},t),
\qquad
\mathbf{x}\in\Omega_{\Delta x}.
$$

Thus, during each time step, the distribution associated with velocity $\mathbf{c}_i$ is transported from one lattice node to another. We use lattice units

$$
\Delta x=\Delta t=1.
$$

In addition to the initial condition, boundary conditions are imposed on $\partial\Omega$. We consider three types:

- **Periodic boundary conditions:** A population leaving the domain through one boundary re-enters through the opposite boundary.
- **Simple-slip boundary conditions:** The velocity component normal to the boundary is reversed, while the tangential components are preserved.
- **Bounce-back boundary conditions:** The complete velocity vector is reversed,

$$
\mathbf{c}_{i,\mathrm{mod}}=-\mathbf{c}_i.
$$

These equations define the classical transport model that we implement in the following sections.

## Implementation setup

Throughout the tutorial, the simulation setup is stored in an `LBMConfig` object.
The configuration contains the information required to construct the lattice and
the corresponding quantum registers, such as the spatial dimension, domain size,
discrete velocity set, boundary conditions, and initial condition.

Spatial regions are described using the `RegionOfInterest` class. A
`RegionOfInterest` specifies a rectangular subregion of the computational domain
through its bounds in each spatial dimension. In this tutorial, it is used to
define the region in which the initial particle distribution is prepared.

For example, in two dimensions a region

$$
\Omega_0=[x_{\min},x_{\max})\times[y_{\min},y_{\max})
$$

defines a rectangular set of lattice nodes that can be used as the initial
distribution.

Keeping the lattice parameters, boundary conditions, and initial condition in the
configuration allows the same QLBM implementation to be used for different one-,
two-, and three-dimensional setups.

In the following sections, we therefore pass `config` to the individual components
of the solver whenever information about the underlying LBM setup is required.

## Quantum transport solver

We now implement the transport problem introduced above as a quantum algorithm using Qrisp.
The particle distribution is encoded as a quantum superposition over lattice positions and
discrete velocities and initialized according to the prescribed initial condition.

During each time step, a **streaming operation** updates the position registers according
to the encoded velocity. Since the position registers use modular arithmetic, this
operation directly implements periodic boundary conditions.

For non-periodic boundaries, auxiliary qubits identify states that cross the physical
boundary. These qubits control the corresponding simple-slip or bounce-back correction
and are subsequently uncomputed so that they return to their initial state.

The resulting solver therefore consists of four main components:

1. **Data encoding:** represent lattice positions, discrete velocities, and the particle
   distribution using quantum registers.
2. **Initial-state preparation:** initialize the registers according to the prescribed
   particle distribution.
3. **Streaming:** propagate each population according to its discrete velocity.
4. **Boundary treatment:** detect and correct states crossing non-periodic boundaries.

After the prescribed number of time steps, the resulting quantum state encodes the
evolved particle distribution.

## Data encoding

The quantum state encodes the three quantities required by the LBM: the lattice position,
the discrete velocity, and the particle distribution.

Let

$$
\mathbf{x}=(x_1,\ldots,x_d)^T \in \Omega_{\Delta x}
$$

denote a lattice node and

$$
\mathbf{c}_j=(c_{j1},\ldots,c_{jd})^T,
\qquad j\in\{1,\ldots,q\},
$$

a discrete velocity from the corresponding DdQq velocity set.

For each spatial dimension $i$, the position $x_i$ is stored in a quantum register
$|x_i\rangle$. Each velocity component $c_{ji}$ is decomposed into its sign
$s_i(c_{ji})$ and magnitude $v_i(c_{ji})$, which are stored in separate quantum
registers.

A position-velocity configuration is therefore represented by

$$
|\mathbf{x}j\rangle
=
\bigotimes_{i=1}^{d}
|x_i\rangle |s_i(c_{ji})\rangle |v_i(c_{ji})\rangle .
$$

The complete particle distribution is encoded in the amplitudes of a superposition over
these configurations,

$$
|\Psi(t)\rangle
=
\sum_{\mathbf{x}\in\Omega_{\Delta x}}
\sum_{j=1}^{q}
\alpha_{\mathbf{x}j}(t)
|\mathbf{x}j\rangle ,
$$

with

$$
f_{\mathbf{x}j}(t)=|\alpha_{\mathbf{x}j}(t)|^2.
$$

Thus, measuring the position and velocity registers samples the discrete particle
distribution represented by the LBM state.

For a lattice with $N_x=2^n$ nodes in each spatial direction, each position coordinate
requires only

$$
n=\log_2 N_x
$$

qubits. The same representation naturally extends from D1Q3 to velocity sets such as
D2Q9 and D3Q27 by introducing the corresponding position and velocity registers for
each spatial dimension.

## Streaming step

The streaming step implements the transport relation introduced above as a unitary
permutation of the position-velocity basis states. For a configuration
$|\mathbf{x}j\rangle$, the position is shifted according to the encoded discrete
velocity $\mathbf{c}_j$, while the velocity registers remain unchanged.

Since the position and velocity components are encoded separately for each spatial
dimension, streaming can be applied component-wise. For each
$i\in\{1,\ldots,d\}$, we define

$$
U_i:
|x_i\rangle |s_i(c_{ji})\rangle |v_i(c_{ji})\rangle
\mapsto
|x_i'\rangle |s_i(c_{ji})\rangle |v_i(c_{ji})\rangle,
$$

where

$$
|x_i'\rangle =
\begin{cases}
|x_i \oplus_{N_x} v_i(c_{ji})\rangle,
& \text{if } |s_i(c_{ji})\rangle = |1\rangle, \\[4pt]
|x_i \ominus_{N_x} v_i(c_{ji})\rangle,
& \text{if } |s_i(c_{ji})\rangle = |0\rangle.
\end{cases}
$$

Here, $\oplus_{N_x}$ and $\ominus_{N_x}$ denote modular addition and subtraction
with modulus $N_x$. The complete streaming operator is obtained by applying this
update to all spatial dimensions.

Because modular addition and subtraction are permutations of the computational
basis, the streaming operation is reversible. Moreover, the modular position
registers naturally implement **periodic boundary conditions**: a state leaving
one side of the lattice re-enters through the opposite side.

In Qrisp, this operation can be implemented directly using controlled quantum
arithmetic:

In [ ]:
from qrisp import QuantumBool, QuantumFloat, QuantumModulus
from qlbm.config.model import LBMConfig


def streaming_step(
    config: LBMConfig,
    position_register: list[QuantumModulus],
    velocity_register: list[QuantumFloat],
    sign_register: list[QuantumBool],
) -> None:
    """Perform one LBM streaming step."""

    for d in range(config.dimensions):
        # Positive direction
        with sign_register[d]:
            position_register[d] += velocity_register[d]

        # Negative direction
        with ~sign_register[d]:
            position_register[d] -= velocity_register[d]

For each spatial dimension, the sign register controls whether the velocity magnitude
is added to or subtracted from the corresponding position register. The use of
`QuantumModulus` ensures that both operations are performed modulo the lattice size.

Non-periodic boundaries therefore require an additional boundary treatment, which
we introduce in the next section.


## Boundary conditions

Periodic boundary conditions are already handled by the modular position registers:
a population leaving one side of the lattice re-enters through the opposite side.

For **simple-slip** and **bounce-back** boundaries, this periodic wrap-around must
instead be detected and corrected. The physical domain is therefore surrounded by
a padding region, also called a ghost-cell region. After streaming, populations
located in this region are identified using auxiliary boundary qubits.

For each non-periodic spatial dimension $i$, a boundary qubit $|b_i\rangle$ indicates
whether the corresponding boundary has been crossed,

$$
|b_i\rangle =
\begin{cases}
|1\rangle, & \text{if a boundary was crossed in dimension } i,\\
|0\rangle, & \text{otherwise}.
\end{cases}
$$

The boundary flags then control how the velocity is reflected.

For a **simple-slip boundary**, only the velocity component pointing towards the
boundary is reversed, while motion parallel to the boundary remains unchanged.

For example, consider a horizontal wall and a population with velocity

$$
\mathbf{c}_j=(c_x,c_y).
$$

The wall reverses the vertical component but preserves the horizontal component,

$$
(c_x,c_y)\longrightarrow(c_x,-c_y).
$$

The population therefore slides along the wall after the reflection.

For a **bounce-back boundary**, all velocity components are reversed,

$$
(c_x,c_y)\longrightarrow(-c_x,-c_y).
$$

The population therefore travels back in the direction from which it arrived.

After the velocity correction, the population is moved from the padding region
back into the physical domain using the same modular arithmetic employed during
streaming.

Finally, the auxiliary boundary qubits are uncomputed. Since the boundary correction
changes both position and velocity, this requires determining from the corrected
state whether the population originated from a boundary. The boundary ancillas can
then be returned to $|0\rangle$ and safely reused.

The implementation supports periodic, simple-slip, bounce-back, and combinations
of these boundary conditions.

## State initialization

The initial quantum state consists of the position and velocity registers introduced
above. In the current implementation, the initial particle distribution is chosen
to be uniform over a rectangular subregion of the computational domain.

Let

$$
\Omega_0 \subseteq \Omega_{\Delta x}
$$

denote this initial region. The position registers are prepared such that all lattice
points inside $\Omega_0$ have equal amplitudes, while states outside the region have
zero amplitude. After normalization, the position state can be written as

$$
|\Psi_x\rangle
=
\frac{1}{\sqrt{|\Omega_0|}}
\sum_{\mathbf{x}\in\Omega_0}
|\mathbf{x}\rangle .
$$

Since the initial region is rectangular, the state can be prepared independently
for each spatial dimension. In Qrisp, the corresponding state vector is constructed
classically and passed to `prepare`:


In [ ]:
def _initialize_subgrid(
    config: LBMConfig,
    position_register: list[QuantumModulus],
    roi: RegionOfInterest,
) -> None:
    """Prepare a uniform superposition over the initial subgrid."""

    encoded_bounds = config.get_encoded_bounds(roi.bounds)

    for d in range(config.dimensions):
        num_states = 2 ** position_register[d].size
        statevector = np.zeros(num_states)

        start, end = encoded_bounds[d]
        statevector[start:end] = 1.0

        statevector /= np.linalg.norm(statevector)

        prepare(position_register[d], statevector)

The initial velocity is specified separately for every spatial dimension. Its
absolute value is stored in the velocity register, while the sign is stored in the
corresponding sign qubit.

For an initial velocity component $u_i$, the registers therefore encode

$$
v_i = |u_i|,
$$

and

$$
s_i =
\begin{cases}
1, & u_i \geq 0,\\
0, & u_i < 0.
\end{cases}
$$

For example, an initial velocity

$$
\mathbf{u}=(1,-1)
$$

prepares velocity magnitudes $(1,1)$ together with a positive sign in the first
dimension and a negative sign in the second dimension.

The complete initialization first prepares the spatial distribution and then the
velocity and sign registers. The resulting state provides the initial condition
for the subsequent streaming and boundary-condition steps.

# Using the QLBM framework

We now combine the individual components introduced above in a complete QLBM
simulation. The high-level `LBMGrid` interface constructs the quantum registers,
initializes the state, applies streaming and boundary conditions, and provides
methods for measuring and visualizing the result.

In this example, we simulate an initially localized distribution on a
two-dimensional D2Q9 lattice.

## Imports

We first import the configuration objects and the high-level components required
to construct and run the simulation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qlbm.api.grid import LBMGrid
from qlbm.config.enums import BoundaryCondition, VelocitySet
from qlbm.config.roi import RegionOfInterest

## Constructing the grid

We first create an $8\times8$ lattice using the D2Q9 velocity set. The boundary
conditions are specified separately for each spatial dimension.

In this example, the first dimension is periodic, while the second dimension uses
bounce-back boundaries.

In [ ]:
grid = LBMGrid(
    grid_size=[8, 8],
    velocity_set=VelocitySet.D2Q9,
    boundary_conditions=[
        BoundaryCondition.PERIODIC,
        BoundaryCondition.BOUNCE_BACK,
    ],
)

## Initial condition

The spatial support of the initial distribution is defined by a
`RegionOfInterest`. The bounds are given separately for each spatial dimension.

We initialize the distribution in the rectangular region

$$
\Omega_0=[2,3)\times[0,1)
$$

with the velocity

$$
\mathbf{u}=(1,-1).
$$

The positive first component transports the distribution in the positive
$x$-direction, while the negative second component transports it in the negative
$y$-direction.


In [ ]:
initial_bubble = RegionOfInterest.from_bounds(
    bounds=[
        (2, 3),
        (0, 1),
    ]
)

initial_velocity = [1, -1]

grid.set_initial_state(
    initial_bubble,
    initial_velocity,
)

## Running the simulation

The method `run_simulation` initializes the quantum state and performs the requested
number of QLBM time steps. Each time step consists of streaming followed by the
configured boundary treatment.

Setting `init=True` instructs the framework to prepare the configured initial state
before starting the simulation.

In [ ]:
grid.run_simulation(
    iterations=2,
    init=True,
)

## Measuring the density

After the final time step, `measure_density` measures the position registers and
aggregates the result into the physical density distribution.

The measurement result is initially returned in the framework's encoded form.
The method `physical_density_to_numpy` converts it into a NumPy array indexed by
the physical lattice coordinates.

In [ ]:
measurement = grid.measure_density()
density = grid.physical_density_to_numpy(measurement)

print(density)

## Visualizing the result

After the simulation, the measured density is converted into a NumPy array using
`physical_density_to_numpy`. The QLBM plotting utility can then directly visualize
the spatial probability distribution as a heatmap.

In [ ]:
from qlbm.io.ploting import plot_res

measurement = grid.measure_density()
density = grid.physical_density_to_numpy(measurement)

plot_res(
    density,
    figsize=(8, 6),
    fontsize=12,
)

## Complete workflow

Putting everything together, a complete QLBM simulation from the lattice
configuration to the final visualization requires only a few high-level calls:

In [ ]:
import numpy as np

from qlbm.api.grid import LBMGrid
from qlbm.config.enums import BoundaryCondition, VelocitySet
from qlbm.config.roi import RegionOfInterest
from qlbm.io.ploting import plot_res


# --------------------------------------------------
# 1. Construct the lattice
# --------------------------------------------------

grid = LBMGrid(
    grid_size=[8, 8],
    velocity_set=VelocitySet.D2Q9,
    boundary_conditions=[
        BoundaryCondition.PERIODIC,
        BoundaryCondition.BOUNCE_BACK,
    ],
)


# --------------------------------------------------
# 2. Define the initial condition
# --------------------------------------------------

initial_bubble = RegionOfInterest.from_bounds(
    bounds=[
        (2, 3),
        (0, 1),
    ]
)

initial_velocity = [1, -1]

grid.set_initial_state(
    initial_bubble,
    initial_velocity,
)


# --------------------------------------------------
# 3. Run the simulation
# --------------------------------------------------

grid.run_simulation(
    iterations=2,
    init=True,
)


# --------------------------------------------------
# 4. Measure the final density
# --------------------------------------------------

measurement = grid.measure_density()
density = grid.physical_density_to_numpy(measurement)


# --------------------------------------------------
# 5. Visualize the result
# --------------------------------------------------

plot_res(
    density,
    figsize=(8, 6),
    fontsize=12,
)

## Visualizing the time evolution

Instead of measuring only the final state, the framework can also export the density
at regular intervals during the simulation. This produces a VTK time series that
can be opened, for example, in ParaView to visualize the evolution of the particle
distribution.

This workflow is particularly useful for inspecting and validating the simulation.
Note, however, that obtaining each frame requires a measurement of the quantum state.
It should therefore be regarded primarily as a visualization and debugging tool,
rather than the intended execution mode of a quantum algorithm.


In [ ]:
from pathlib import Path

from qlbm.api.grid import LBMGrid
from qlbm.config.enums import BoundaryCondition, VelocitySet
from qlbm.config.roi import RegionOfInterest
from qlbm.io.vtk import clear_vtk_series


# --------------------------------------------------
# 1. Construct the lattice
# --------------------------------------------------

grid = LBMGrid(
    grid_size=[32, 8],
    velocity_set=VelocitySet.D2Q9,
    boundary_conditions=[
        BoundaryCondition.PERIODIC,
        BoundaryCondition.SIMPLE_SLIP,
    ],
)


# --------------------------------------------------
# 2. Define the initial condition
# --------------------------------------------------

initial_bubble = RegionOfInterest.from_bounds(
    bounds=[
        (1, 4),
        (2, 6),
    ]
)

grid.set_initial_state(
    initial_bubble,
    initial_velocity=[1, -1],
)


# --------------------------------------------------
# 3. Prepare the output directory
# --------------------------------------------------

output_path = Path("output") / "2d_visualization"
clear_vtk_series(output_path)


# --------------------------------------------------
# 4. Run and export the time evolution
# --------------------------------------------------

grid.run_simulation_with_visualization(
    output_path,
    iterations=40,
    init=True,
    time_steps_per_frame=1,
)